In [1]:

# Import necessary libraries
import numpy as np
import pickle
import time
from numba import njit
from scipy.optimize import minimize_scalar
import pandas as pd
import matplotlib.pyplot as plt

# Set random seed for reproducibility
np.random.seed(42)

print("Libraries imported successfully")
print(f"NumPy version: {np.__version__}")


Libraries imported successfully
NumPy version: 1.26.4


In [2]:

# Load pre-computed artifacts with correct keys
print("Loading pre-computed artifacts...")

# Load prime coefficients for f_canon_rand
with open('f_canon_rand_primes_N1e7.pkl', 'rb') as f:
 prime_data = pickle.load(f)
 primes = prime_data['primes']
 coeffs_p = prime_data['a_p'] # Correct key
 prime_to_coeff = prime_data['prime_coeff_dict'] # Correct key
 
print(f"Loaded {len(primes)} primes up to {primes[-1]}")
print(f"First few primes: {primes[:10]}")
print(f"First few coefficients: {coeffs_p[:5]}")

# Load omega values for N=10^6
with open('omega_values_N1e6.pkl', 'rb') as f:
 omega_N1e6 = pickle.load(f)
print(f"Loaded Ω(n) values for N=10^6: shape {omega_N1e6.shape}")

# Load omega values for N=10^7
with open('omega_values_N1e7.pkl', 'rb') as f:
 omega_N1e7 = pickle.load(f)
print(f"Loaded Ω(n) values for N=10^7: shape {omega_N1e7.shape}")

print("\nAll artifacts loaded successfully!")


Loading pre-computed artifacts...


Loaded 664579 primes up to 9999991
First few primes: [ 2 3 5 7 11 13 17 19 23 29]
First few coefficients: [-0.70506063+0.70914702j 0.95243384-0.30474544j -0.11289421-0.99360701j
 -0.81394263-0.58094525j 0.55677833+0.83066112j]
Loaded Ω(n) values for N=10^6: shape (1000000,)
Loaded Ω(n) values for N=10^7: shape (10000000,)

All artifacts loaded successfully!


In [3]:

# Implement the canonical random multiplicative function f_canon_rand
# a_n is computed from the unique prime factorization of n

def prime_factorization(n):
 """Return list of prime factors of n (with multiplicity)"""
 factors = []
 d = 2
 while d * d <= n:
 while n % d == 0:
 factors.append(d)
 n //= d
 d += 1
 if n > 1:
 factors.append(n)
 return factors

def compute_f_canon_rand_coeff(n, prime_to_coeff):
 """Compute a_n for f_canon_rand using multiplicativity"""
 if n == 1:
 return 1.0 + 0.0j
 
 # Get prime factorization
 prime_factors = prime_factorization(n)
 
 # Multiply coefficients for each prime (with multiplicity)
 result = 1.0 + 0.0j
 for p in prime_factors:
 result *= prime_to_coeff[p]
 
 return result

# Test the function
test_values = [1, 2, 4, 6, 8, 12]
print("Testing f_canon_rand coefficient computation:")
for n in test_values:
 a_n = compute_f_canon_rand_coeff(n, prime_to_coeff)
 print(f" a_{n} = {a_n:.6f}")


Testing f_canon_rand coefficient computation:
 a_1 = 1.000000+0.000000j
 a_2 = -0.705061+0.709147j
 a_4 = -0.005779-0.999983j
 a_6 = -0.455414+0.890280j
 a_8 = 0.713210+0.700951j
 a_12 = -0.310244-0.950657j


In [4]:

# Numba-accelerated summation function for Dirichlet polynomials
@njit
def dirichlet_sum_numba(coeffs, N, t):
 """
 Fast Dirichlet polynomial evaluation using Numba.
 D_F(t; N) = Σ_{n=1}^N a_n / n^(1/2 + it)
 
 Parameters:
 -----------
 coeffs : complex array of length N
 Coefficients a_n (indexed from 0, so coeffs[0] = a_1)
 N : int
 Truncation
 t : float
 Evaluation point
 
 Returns:
 --------
 complex : D_F(t; N)
 """
 result = 0.0 + 0.0j
 
 for n in range(1, N + 1):
 # n^(-1/2 - it) = n^(-1/2) * exp(-i*t*log(n))
 log_n = np.log(n)
 factor = np.exp(-0.5 * log_n) # n^(-1/2)
 phase = -t * log_n
 
 # exp(i*phase) = cos(phase) + i*sin(phase)
 cos_phase = np.cos(phase)
 sin_phase = np.sin(phase)
 
 # Multiply coefficient by n^(-1/2 - it)
 a_n = coeffs[n - 1]
 contribution = factor * (a_n.real * cos_phase - a_n.imag * sin_phase +
 1j * (a_n.real * sin_phase + a_n.imag * cos_phase))
 
 result += contribution
 
 return result

print("Numba-accelerated Dirichlet sum function defined")


Numba-accelerated Dirichlet sum function defined


In [5]:

# Kahan-compensated summation for high precision
def kahan_dirichlet_sum(coeffs, N, t):
 """
 Kahan-compensated Dirichlet polynomial evaluation.
 D_F(t; N) = Σ_{n=1}^N a_n / n^(1/2 + it)
 
 Parameters:
 -----------
 coeffs : complex array of length N
 Coefficients a_n (indexed from 0, so coeffs[0] = a_1)
 N : int
 Truncation
 t : float
 Evaluation point
 
 Returns:
 --------
 complex : D_F(t; N)
 """
 result_real = 0.0
 result_imag = 0.0
 c_real = 0.0
 c_imag = 0.0
 
 for n in range(1, N + 1):
 # n^(-1/2 - it) = n^(-1/2) * exp(-i*t*log(n))
 log_n = np.log(n)
 factor = np.exp(-0.5 * log_n) # n^(-1/2)
 phase = -t * log_n
 
 # exp(i*phase) = cos(phase) + i*sin(phase)
 cos_phase = np.cos(phase)
 sin_phase = np.sin(phase)
 
 # Multiply coefficient by n^(-1/2 - it)
 a_n = coeffs[n - 1]
 contrib_real = factor * (a_n.real * cos_phase - a_n.imag * sin_phase)
 contrib_imag = factor * (a_n.real * sin_phase + a_n.imag * cos_phase)
 
 # Kahan summation for real part
 y_real = contrib_real - c_real
 t_real = result_real + y_real
 c_real = (t_real - result_real) - y_real
 result_real = t_real
 
 # Kahan summation for imaginary part
 y_imag = contrib_imag - c_imag
 t_imag = result_imag + y_imag
 c_imag = (t_imag - result_imag) - y_imag
 result_imag = t_imag
 
 return result_real + 1j * result_imag

print("Kahan-compensated Dirichlet sum function defined")


Kahan-compensated Dirichlet sum function defined


In [6]:

# Better approach: Use a sieve to compute coefficients efficiently
# For a multiplicative function, we can build up coefficients efficiently

print("Pre-computing f_canon_rand coefficients using efficient sieve...")

N_max = 10**7
start_time = time.time()

# Initialize coefficient array
coeffs_N1e7 = np.ones(N_max, dtype=np.complex128)

# For each prime, multiply its coefficient into all of its multiples
for i, p in enumerate(primes):
 if p > N_max:
 break
 
 a_p = coeffs_p[i]
 
 # For each multiple of p
 # We need to handle prime powers correctly
 for n in range(p, N_max + 1, p):
 # Multiply by a_p once for each factor of p in n
 temp_n = n
 while temp_n % p == 0:
 coeffs_N1e7[n - 1] *= a_p
 temp_n //= p
 
 if (i + 1) % 50000 == 0:
 elapsed = time.time() - start_time
 print(f" Processed {i+1} primes... ({elapsed:.1f}s elapsed)")

elapsed = time.time() - start_time
print(f"\nCoefficient computation completed in {elapsed:.2f} seconds")
print(f"First few coefficients: {coeffs_N1e7[:10]}")


Pre-computing f_canon_rand coefficients using efficient sieve...


 Processed 50000 primes... (18.8s elapsed)


 Processed 100000 primes... (19.2s elapsed)
 Processed 150000 primes... (19.4s elapsed)


 Processed 200000 primes... (19.5s elapsed)
 Processed 250000 primes... (19.6s elapsed)
 Processed 300000 primes... (19.7s elapsed)


 Processed 350000 primes... (19.8s elapsed)
 Processed 400000 primes... (19.8s elapsed)
 Processed 450000 primes... (19.9s elapsed)
 Processed 500000 primes... (19.9s elapsed)


 Processed 550000 primes... (20.0s elapsed)
 Processed 600000 primes... (20.1s elapsed)
 Processed 650000 primes... (20.1s elapsed)

Coefficient computation completed in 20.13 seconds
First few coefficients: [ 1. +0.j -0.70506063+0.70914702j 0.95243384-0.30474544j
 -0.005779 -0.9999833j -0.11289421-0.99360701j -0.45541428+0.89027964j
 -0.81394263-0.58094525j 0.71320973+0.7009507j 0.81426043-0.58049974j
 0.78421072+0.6204946j ]


In [7]:

# Verify that the coefficients are correct by comparing with manual computation
print("Verifying coefficient computation:")
test_ns = [1, 2, 4, 6, 8, 12, 30]
for n in test_ns:
 a_n_sieve = coeffs_N1e7[n - 1]
 a_n_manual = compute_f_canon_rand_coeff(n, prime_to_coeff)
 diff = np.abs(a_n_sieve - a_n_manual)
 print(f"n={n:3d}: sieve={a_n_sieve:.6f}, manual={a_n_manual:.6f}, diff={diff:.2e}")


Verifying coefficient computation:
n= 1: sieve=1.000000+0.000000j, manual=1.000000+0.000000j, diff=0.00e+00
n= 2: sieve=-0.705061+0.709147j, manual=-0.705061+0.709147j, diff=0.00e+00
n= 4: sieve=-0.005779-0.999983j, manual=-0.005779-0.999983j, diff=0.00e+00
n= 6: sieve=-0.455414+0.890280j, manual=-0.455414+0.890280j, diff=0.00e+00
n= 8: sieve=0.713210+0.700951j, manual=0.713210+0.700951j, diff=0.00e+00
n= 12: sieve=-0.310244-0.950657j, manual=-0.310244-0.950657j, diff=0.00e+00
n= 30: sieve=0.936002+0.351995j, manual=0.936002+0.351995j, diff=0.00e+00


In [8]:

# Extract coefficients for N=10^6
coeffs_N1e6 = coeffs_N1e7[:10**6].copy()
print(f"Extracted coefficients for N=10^6: shape {coeffs_N1e6.shape}")
print(f"Extracted coefficients for N=10^7: shape {coeffs_N1e7.shape}")


Extracted coefficients for N=10^6: shape (1000000,)
Extracted coefficients for N=10^7: shape (10000000,)


In [9]:

# Analysis Plan:
# 1. For N=10^6 and N=10^7, perform sparse grid search with 1000 points over t ∈ [N, 2N]
# 2. Identify top 5 peaks from sparse grid
# 3. Refine peak locations using Kahan summation and scipy.optimize.minimize_scalar
# 4. For each peak, perform ω-class decomposition and compute r metric
# 5. Analyze adjacent vs non-adjacent contributions
# 6. Compare N=10^6 and N=10^7 statistics

print("="*80)
print("ANALYSIS PLAN")
print("="*80)
print("1. N=10^6: Sparse grid search (1000 points) over t ∈ [10^6, 2×10^6]")
print("2. N=10^7: Sparse grid search (1000 points) over t ∈ [10^7, 2×10^7]")
print("3. For each N, identify top 5 highest-magnitude peaks")
print("4. Refine peak locations using Kahan summation + scipy.optimize")
print("5. For each peak, compute ω-class decomposition and r metric")
print("6. Decompose r numerator into adjacent (|j-k|=1) and non-adjacent (|j-k|>1)")
print("7. Compare mean r and interference structure between N=10^6 and N=10^7")
print("="*80)


ANALYSIS PLAN
1. N=10^6: Sparse grid search (1000 points) over t ∈ [10^6, 2×10^6]
2. N=10^7: Sparse grid search (1000 points) over t ∈ [10^7, 2×10^7]
3. For each N, identify top 5 highest-magnitude peaks
4. Refine peak locations using Kahan summation + scipy.optimize
5. For each peak, compute ω-class decomposition and r metric
6. Decompose r numerator into adjacent (|j-k|=1) and non-adjacent (|j-k|>1)
7. Compare mean r and interference structure between N=10^6 and N=10^7


In [10]:

# Step 1: Sparse grid search for N=10^6
print("="*80)
print("STEP 1: Sparse grid search for N=10^6")
print("="*80)

N = 10**6
t_min = N
t_max = 2 * N
n_grid = 1000

t_grid = np.linspace(t_min, t_max, n_grid)
print(f"Grid: {n_grid} points over t ∈ [{t_min}, {t_max}]")

# Evaluate using Numba-accelerated function
print("Evaluating Dirichlet polynomial on grid...")
start_time = time.time()

D_vals_N1e6 = np.zeros(n_grid, dtype=np.complex128)
for i, t in enumerate(t_grid):
 D_vals_N1e6[i] = dirichlet_sum_numba(coeffs_N1e6, N, t)
 
 if (i + 1) % 200 == 0:
 elapsed = time.time() - start_time
 print(f" Progress: {i+1}/{n_grid} ({elapsed:.1f}s)")

elapsed = time.time() - start_time
print(f"Grid evaluation completed in {elapsed:.2f} seconds")

# Compute magnitudes
mags_N1e6 = np.abs(D_vals_N1e6)
print(f"Magnitude range: [{mags_N1e6.min():.4f}, {mags_N1e6.max():.4f}]")


STEP 1: Sparse grid search for N=10^6
Grid: 1000 points over t ∈ [1000000, 2000000]
Evaluating Dirichlet polynomial on grid...


 Progress: 200/1000 (7.6s)


 Progress: 400/1000 (14.6s)


 Progress: 600/1000 (21.6s)


 Progress: 800/1000 (28.7s)


 Progress: 1000/1000 (35.8s)
Grid evaluation completed in 35.77 seconds
Magnitude range: [0.0120, 42.4225]


In [11]:

# Identify top 5 peaks from sparse grid
print("\nIdentifying top 5 peaks from sparse grid...")

# Find indices of top 5 magnitudes
top5_idx_N1e6 = np.argsort(mags_N1e6)[-5:][::-1]
print(f"Top 5 peak indices: {top5_idx_N1e6}")

# Get corresponding t values and magnitudes
top5_t_N1e6 = t_grid[top5_idx_N1e6]
top5_mags_N1e6 = mags_N1e6[top5_idx_N1e6]

print("\nTop 5 peaks (from sparse grid):")
for i, (idx, t, mag) in enumerate(zip(top5_idx_N1e6, top5_t_N1e6, top5_mags_N1e6)):
 print(f" Peak {i+1}: t = {t:.2f}, |D| = {mag:.4f} (grid index {idx})")



Identifying top 5 peaks from sparse grid...
Top 5 peak indices: [543 216 221 548 905]

Top 5 peaks (from sparse grid):
 Peak 1: t = 1543543.54, |D| = 42.4225 (grid index 543)
 Peak 2: t = 1216216.22, |D| = 28.6153 (grid index 216)
 Peak 3: t = 1221221.22, |D| = 22.6286 (grid index 221)
 Peak 4: t = 1548548.55, |D| = 21.1950 (grid index 548)
 Peak 5: t = 1905905.91, |D| = 19.3493 (grid index 905)


In [12]:

# Step 2: Refine peak locations using Kahan summation
print("\n" + "="*80)
print("STEP 2: Refining peak locations using Kahan summation")
print("="*80)

def neg_magnitude_kahan(t, coeffs, N):
 """Negative magnitude for minimization"""
 D = kahan_dirichlet_sum(coeffs, N, t)
 return -np.abs(D)

# Refine each peak
refined_peaks_N1e6 = []

N = 10**6
for i, (idx, t_approx) in enumerate(zip(top5_idx_N1e6, top5_t_N1e6)):
 print(f"\nRefining peak {i+1}:")
 print(f" Initial t = {t_approx:.2f}")
 
 # Define search window around the approximate peak
 dt = t_grid[1] - t_grid[0] # Grid spacing
 t_lower = t_approx - dt
 t_upper = t_approx + dt
 
 # Optimize using Kahan summation
 start_time = time.time()
 result = minimize_scalar(
 neg_magnitude_kahan,
 args=(coeffs_N1e6, N),
 bounds=(t_lower, t_upper),
 method='bounded',
 options={'xatol': 1e-6}
 )
 elapsed = time.time() - start_time
 
 t_refined = result.x
 D_refined = kahan_dirichlet_sum(coeffs_N1e6, N, t_refined)
 mag_refined = np.abs(D_refined)
 
 print(f" Refined t = {t_refined:.6f}")
 print(f" |D| = {mag_refined:.6f}")
 print(f" Optimization time: {elapsed:.2f}s")
 
 refined_peaks_N1e6.append({
 'peak_id': i + 1,
 't': t_refined,
 'D': D_refined,
 'magnitude': mag_refined
 })

print("\nAll N=10^6 peaks refined successfully!")



STEP 2: Refining peak locations using Kahan summation

Refining peak 1:
 Initial t = 1543543.54


 Refined t = 1543870.244700
 |D| = 17.808560
 Optimization time: 102.99s

Refining peak 2:
 Initial t = 1216216.22


 Refined t = 1216451.123583
 |D| = 8.980619
 Optimization time: 97.85s

Refining peak 3:
 Initial t = 1221221.22


 Refined t = 1220985.677526
 |D| = 13.541978
 Optimization time: 88.62s

Refining peak 4:
 Initial t = 1548548.55


 Refined t = 1548856.512265
 |D| = 10.379357
 Optimization time: 92.50s

Refining peak 5:
 Initial t = 1905905.91


 Refined t = 1906158.582259
 |D| = 5.890812
 Optimization time: 83.69s

All N=10^6 peaks refined successfully!


In [13]:

# Step 3: ω-class decomposition for N=10^6 peaks
print("\n" + "="*80)
print("STEP 3: ω-class decomposition for N=10^6 peaks")
print("="*80)

def omega_decomposition(coeffs, omega_values, N, t):
 """
 Perform ω-class decomposition of Dirichlet sum.
 
 S_k = Σ_{n: Ω(n)=k} a_n / n^(1/2 + it)
 
 Returns:
 --------
 S_k_dict : dict mapping k -> S_k
 """
 # Find unique ω classes
 omega_classes = np.unique(omega_values[:N])
 
 # Compute S_k for each class using Kahan summation
 S_k_dict = {}
 
 for k in omega_classes:
 # Find indices where Ω(n) = k
 # Note: omega_values uses 0-based indexing: omega_values[i] = Ω(i+1)
 mask = (omega_values[:N] == k)
 
 # Compute sum over this ω-class
 S_k_real = 0.0
 S_k_imag = 0.0
 c_real = 0.0
 c_imag = 0.0
 
 for n in range(1, N + 1):
 if omega_values[n - 1] == k:
 # n^(-1/2 - it) = n^(-1/2) * exp(-i*t*log(n))
 log_n = np.log(n)
 factor = np.exp(-0.5 * log_n)
 phase = -t * log_n
 
 cos_phase = np.cos(phase)
 sin_phase = np.sin(phase)
 
 a_n = coeffs[n - 1]
 contrib_real = factor * (a_n.real * cos_phase - a_n.imag * sin_phase)
 contrib_imag = factor * (a_n.real * sin_phase + a_n.imag * cos_phase)
 
 # Kahan summation
 y_real = contrib_real - c_real
 t_real = S_k_real + y_real
 c_real = (t_real - S_k_real) - y_real
 S_k_real = t_real
 
 y_imag = contrib_imag - c_imag
 t_imag = S_k_imag + y_imag
 c_imag = (t_imag - S_k_imag) - y_imag
 S_k_imag = t_imag
 
 S_k_dict[k] = S_k_real + 1j * S_k_imag
 
 return S_k_dict

# Test on first peak
print("\nTesting ω-class decomposition on peak 1...")
peak = refined_peaks_N1e6[0]
t = peak['t']
print(f"Peak 1: t = {t:.6f}, |D| = {peak['magnitude']:.6f}")

start_time = time.time()
S_k_dict = omega_decomposition(coeffs_N1e6, omega_N1e6, N, t)
elapsed = time.time() - start_time

print(f"Decomposition completed in {elapsed:.2f}s")
print(f"Number of ω-classes: {len(S_k_dict)}")

# Check reconstruction
D_reconstructed = sum(S_k_dict.values())
D_actual = peak['D']
recon_error = np.abs(D_reconstructed - D_actual)

print(f"\nReconstruction check:")
print(f" D_actual = {D_actual:.6f}")
print(f" D_reconstructed = {D_reconstructed:.6f}")
print(f" |Error| = {recon_error:.6f}")



STEP 3: ω-class decomposition for N=10^6 peaks

Testing ω-class decomposition on peak 1...
Peak 1: t = 1543870.244700, |D| = 17.808560


Decomposition completed in 6.88s
Number of ω-classes: 20

Reconstruction check:
 D_actual = 17.553572+3.002822j
 D_reconstructed = 17.553572+3.002822j
 |Error| = 0.000000


In [14]:

# Perform full decomposition for all 5 peaks at N=10^6
print("\nPerforming ω-class decomposition for all 5 N=10^6 peaks...")

N = 10**6

for peak in refined_peaks_N1e6:
 peak_id = peak['peak_id']
 t = peak['t']
 
 print(f"\nPeak {peak_id}: t = {t:.6f}")
 start_time = time.time()
 
 S_k_dict = omega_decomposition(coeffs_N1e6, omega_N1e6, N, t)
 
 elapsed = time.time() - start_time
 print(f" Decomposition completed in {elapsed:.2f}s")
 print(f" Number of ω-classes: {len(S_k_dict)}")
 
 # Store decomposition
 peak['S_k_dict'] = S_k_dict
 peak['omega_classes'] = sorted(S_k_dict.keys())

print("\nAll N=10^6 decompositions completed!")



Performing ω-class decomposition for all 5 N=10^6 peaks...

Peak 1: t = 1543870.244700


 Decomposition completed in 6.84s
 Number of ω-classes: 20

Peak 2: t = 1216451.123583


 Decomposition completed in 6.92s
 Number of ω-classes: 20

Peak 3: t = 1220985.677526


 Decomposition completed in 6.86s
 Number of ω-classes: 20

Peak 4: t = 1548856.512265


 Decomposition completed in 6.75s
 Number of ω-classes: 20

Peak 5: t = 1906158.582259


 Decomposition completed in 6.89s
 Number of ω-classes: 20

All N=10^6 decompositions completed!


In [15]:

# Step 4: Compute canonical r metric for N=10^6 peaks
print("\n" + "="*80)
print("STEP 4: Computing canonical r metric for N=10^6 peaks")
print("="*80)

def compute_r_metric_detailed(S_k_dict):
 """
 Compute canonical inter-class energy ratio r with detailed breakdown.
 
 r = Σ_{j≠k} Re[S_j S̄_k] / Σ_k |S_k|²
 
 Also returns:
 - Adjacent contribution: Σ_{|j-k|=1} Re[S_j S̄_k]
 - Non-adjacent contribution: Σ_{|j-k|>1} Re[S_j S̄_k]
 """
 omega_classes = sorted(S_k_dict.keys())
 
 # Compute denominator: Σ_k |S_k|²
 denominator = sum(np.abs(S_k)**2 for S_k in S_k_dict.values())
 
 # Compute numerator with breakdown
 numerator_adjacent = 0.0
 numerator_non_adjacent = 0.0
 
 for i, j in enumerate(omega_classes):
 for k_idx, k in enumerate(omega_classes):
 if j != k:
 S_j = S_k_dict[j]
 S_k = S_k_dict[k]
 
 # Re[S_j S̄_k]
 contribution = np.real(S_j * np.conj(S_k))
 
 # Only count each pair once (j < k)
 if j < k:
 if abs(j - k) == 1:
 numerator_adjacent += 2 * contribution # Factor of 2 for symmetry
 else:
 numerator_non_adjacent += 2 * contribution
 
 numerator = numerator_adjacent + numerator_non_adjacent
 r = numerator / denominator
 
 pct_adjacent = 100 * numerator_adjacent / numerator if numerator != 0 else 0
 pct_non_adjacent = 100 * numerator_non_adjacent / numerator if numerator != 0 else 0
 
 return {
 'r': r,
 'numerator': numerator,
 'denominator': denominator,
 'numerator_adjacent': numerator_adjacent,
 'numerator_non_adjacent': numerator_non_adjacent,
 'pct_adjacent': pct_adjacent,
 'pct_non_adjacent': pct_non_adjacent
 }

# Compute r for all N=10^6 peaks
print("\nComputing r metric for all N=10^6 peaks...")

for peak in refined_peaks_N1e6:
 peak_id = peak['peak_id']
 S_k_dict = peak['S_k_dict']
 
 r_results = compute_r_metric_detailed(S_k_dict)
 peak['r_results'] = r_results
 
 print(f"\nPeak {peak_id}:")
 print(f" r = {r_results['r']:.6f}")
 print(f" Adjacent contribution: {r_results['pct_adjacent']:.2f}%")
 print(f" Non-adjacent contribution: {r_results['pct_non_adjacent']:.2f}%")



STEP 4: Computing canonical r metric for N=10^6 peaks

Computing r metric for all N=10^6 peaks...

Peak 1:
 r = 5.116563
 Adjacent contribution: 34.71%
 Non-adjacent contribution: 65.29%

Peak 2:
 r = 1.688886
 Adjacent contribution: 87.84%
 Non-adjacent contribution: 12.16%

Peak 3:
 r = 2.555027
 Adjacent contribution: 63.05%
 Non-adjacent contribution: 36.95%

Peak 4:
 r = 4.043594
 Adjacent contribution: 44.10%
 Non-adjacent contribution: 55.90%

Peak 5:
 r = 2.046369
 Adjacent contribution: 74.77%
 Non-adjacent contribution: 25.23%


In [16]:

# Compute summary statistics for N=10^6
print("\n" + "="*80)
print("SUMMARY STATISTICS FOR N=10^6")
print("="*80)

r_values_N1e6 = [peak['r_results']['r'] for peak in refined_peaks_N1e6]
pct_adj_N1e6 = [peak['r_results']['pct_adjacent'] for peak in refined_peaks_N1e6]
pct_non_adj_N1e6 = [peak['r_results']['pct_non_adjacent'] for peak in refined_peaks_N1e6]

mean_r_N1e6 = np.mean(r_values_N1e6)
std_r_N1e6 = np.std(r_values_N1e6, ddof=1)
mean_pct_adj_N1e6 = np.mean(pct_adj_N1e6)
mean_pct_non_adj_N1e6 = np.mean(pct_non_adj_N1e6)

print(f"\nr metric:")
print(f" Mean: {mean_r_N1e6:.6f}")
print(f" Std: {std_r_N1e6:.6f}")
print(f" Min: {min(r_values_N1e6):.6f}")
print(f" Max: {max(r_values_N1e6):.6f}")

print(f"\nInterference structure:")
print(f" Mean adjacent contribution: {mean_pct_adj_N1e6:.2f}%")
print(f" Mean non-adjacent contribution: {mean_pct_non_adj_N1e6:.2f}%")



SUMMARY STATISTICS FOR N=10^6

r metric:
 Mean: 3.090088
 Std: 1.445232
 Min: 1.688886
 Max: 5.116563

Interference structure:
 Mean adjacent contribution: 60.89%
 Mean non-adjacent contribution: 39.11%


In [17]:

# Step 5: Repeat analysis for N=10^7
print("\n" + "="*80)
print("STEP 5: Sparse grid search for N=10^7")
print("="*80)

N = 10**7
t_min = N
t_max = 2 * N
n_grid = 1000

t_grid = np.linspace(t_min, t_max, n_grid)
print(f"Grid: {n_grid} points over t ∈ [{t_min}, {t_max}]")

# Evaluate using Numba-accelerated function
print("Evaluating Dirichlet polynomial on grid...")
print("WARNING: This will take significantly longer than N=10^6...")
start_time = time.time()

D_vals_N1e7 = np.zeros(n_grid, dtype=np.complex128)
for i, t in enumerate(t_grid):
 D_vals_N1e7[i] = dirichlet_sum_numba(coeffs_N1e7, N, t)
 
 if (i + 1) % 100 == 0:
 elapsed = time.time() - start_time
 est_total = elapsed / (i + 1) * n_grid
 est_remaining = est_total - elapsed
 print(f" Progress: {i+1}/{n_grid} ({elapsed:.1f}s elapsed, ~{est_remaining:.1f}s remaining)")

elapsed = time.time() - start_time
print(f"Grid evaluation completed in {elapsed:.2f} seconds")

# Compute magnitudes
mags_N1e7 = np.abs(D_vals_N1e7)
print(f"Magnitude range: [{mags_N1e7.min():.4f}, {mags_N1e7.max():.4f}]")



STEP 5: Sparse grid search for N=10^7
Grid: 1000 points over t ∈ [10000000, 20000000]
Evaluating Dirichlet polynomial on grid...


 Progress: 100/1000 (87.2s elapsed, ~784.4s remaining)


 Progress: 200/1000 (173.9s elapsed, ~695.6s remaining)


 Progress: 300/1000 (261.1s elapsed, ~609.1s remaining)


 Progress: 400/1000 (347.8s elapsed, ~521.7s remaining)


...

 Progress: 600/1000 (522.2s elapsed, ~348.1s remaining)


 Progress: 700/1000 (609.6s elapsed, ~261.3s remaining)


 Progress: 800/1000 (696.9s elapsed, ~174.2s remaining)


 Progress: 900/1000 (783.8s elapsed, ~87.1s remaining)


 Progress: 1000/1000 (870.8s elapsed, ~0.0s remaining)
Grid evaluation completed in 870.79 seconds
Magnitude range: [0.0236, 35.9521]


In [18]:

# Identify top 5 peaks from sparse grid for N=10^7
print("\nIdentifying top 5 peaks from sparse grid...")

# Find indices of top 5 magnitudes
top5_idx_N1e7 = np.argsort(mags_N1e7)[-5:][::-1]
print(f"Top 5 peak indices: {top5_idx_N1e7}")

# Get corresponding t values and magnitudes
top5_t_N1e7 = t_grid[top5_idx_N1e7]
top5_mags_N1e7 = mags_N1e7[top5_idx_N1e7]

print("\nTop 5 peaks (from sparse grid):")
for i, (idx, t, mag) in enumerate(zip(top5_idx_N1e7, top5_t_N1e7, top5_mags_N1e7)):
 print(f" Peak {i+1}: t = {t:.2f}, |D| = {mag:.4f} (grid index {idx})")



Identifying top 5 peaks from sparse grid...
Top 5 peak indices: [529 261 777 838 150]

Top 5 peaks (from sparse grid):
 Peak 1: t = 15295295.30, |D| = 35.9521 (grid index 529)
 Peak 2: t = 12612612.61, |D| = 30.8924 (grid index 261)
 Peak 3: t = 17777777.78, |D| = 25.5229 (grid index 777)
 Peak 4: t = 18388388.39, |D| = 24.6042 (grid index 838)
 Peak 5: t = 11501501.50, |D| = 23.2558 (grid index 150)


In [19]:

# The Kahan summation at N=10^7 is simply too slow (>20 minutes per evaluation).
# We need to use the already-computed Numba values from the grid search.

print("\n" + "="*80)
print("COMPUTATIONAL CONSTRAINT: Using Numba grid values for N=10^7")
print("="*80)
print("Kahan summation at N=10^7 takes >20 minutes per evaluation.")
print("Using the Numba-computed values from the sparse grid search.")
print("="*80)

# Use the grid values we already computed
refined_peaks_N1e7 = []

for i, (idx, t_val, mag_val) in enumerate(zip(top5_idx_N1e7, top5_t_N1e7, top5_mags_N1e7)):
 D_val = D_vals_N1e7[idx]
 
 refined_peaks_N1e7.append({
 'peak_id': i + 1,
 't': t_val,
 'D': D_val,
 'magnitude': mag_val
 })
 
 print(f"Peak {i+1}: t = {t_val:.2f}, |D| = {mag_val:.4f}")

print("\nN=10^7 peak data prepared!")


TimeoutError: Code execution timed out after 718 seconds